# Python

In [1]:
import random
import time

def monitoreo(zona, eventos, pesos, criticos_lista, duracion, frecuencia, cola):
    totales = 0
    criticos = 0
    fin_monitoreo = time.monotonic() + duracion

    while time.monotonic() < fin_monitoreo:
        evento = random.choices(eventos, weights=pesos)[0]
        if evento in criticos_lista:
            criticos += 1
        totales += 1
        print(f"[{zona}] - {evento}")

        tiempo_restante = fin_monitoreo - time.monotonic()
        if tiempo_restante > 0:
            time.sleep(min(frecuencia, tiempo_restante))

    cola.put({
        "zona": zona,
        "totales": totales,
        "criticos": criticos
    })

def recibir_reportes(cola, cantidad):
    return [cola.get() for _ in range(cantidad)]


In [5]:
import multiprocessing

if __name__ == '__main__':
  cola = multiprocessing.Queue()
  duracion_monitoreo = 10
  frecuencia_reporte = 2

  zonas_config = [
    {
        "nombre": "Sector del Tiranosaurio",
        "eventos": ["Todo normal", "Tiranosaurio fuera del recinto", "Falla en el cerco eléctrico"],
        "pesos": [0.8, 0.1, 0.1],
        "criticos": ["Tiranosaurio fuera del recinto", "Falla en el cerco eléctrico"]
    },
    {
        "nombre": "Área de Velociraptores",
        "eventos": ["Todo normal", "Pérdida de visibilidad", "Falla en el cerco eléctrico"],
        "pesos": [0.7, 0.2, 0.1],
        "criticos": ["Falla en el cerco eléctrico"]
    },
    {
        "nombre": "Recinto de los triceratops",
        "eventos": ["Todo normal", "Comportamiento inusual", "Estampida"],
        "pesos": [0.6, 0.3, 0.1],
        "criticos": []
    },
    {
        "nombre": "Centro de Visitantes",
        "eventos": ["Todo normal", "Pérdida de comunicación", "Alerta de seguridad"],
        "pesos": [0.8, 0.15, 0.05],
        "criticos": ["Pérdida de comunicación", "Alerta de seguridad"]
    },
    {
        "nombre": "Laboratorio Genético",
        "eventos": ["Todo normal", "Falla del sistema", "Pérdida de comunicación", "Acceso no autorizado"],
        "pesos": [0.8, 0.1, 0.05, 0.05],
        "criticos": ["Pérdida de comunicación"]
    },
  ]

  procesos = []

  for zona in zonas_config:
    proceso = multiprocessing.Process(
      target=monitoreo,
      args=(zona["nombre"], zona["eventos"], zona["pesos"], zona["criticos"], duracion_monitoreo, frecuencia_reporte, cola)
    )
    proceso.start()
    procesos.append(proceso)

  for proceso in procesos:
    proceso.join()

  print("\n" + "="*70)
  print(f"{'REPORTE FINAL DE MONITOREO JURASSIC PARK':^70}")
  print("="*70)
  print(f"{'Zona':<30} | {'Eventos Totales':^15} | {'Eventos Críticos':^15}")
  print("-"*70)

  for reporte in recibir_reportes(cola, len(procesos)):
    print(f"{reporte['zona']:<30} | {reporte['totales']:^15} | {reporte['criticos']:^15}")

  print("="*70)


[Sector del Tiranosaurio] - Todo normal
[Área de Velociraptores] - Todo normal
[Recinto de los triceratops] - Todo normal[Centro de Visitantes] - Todo normal

[Laboratorio Genético] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Área de Velociraptores] - Todo normal
[Recinto de los triceratops] - Comportamiento inusual
[Centro de Visitantes] - Todo normal
[Laboratorio Genético] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Área de Velociraptores] - Todo normal
[Recinto de los triceratops] - Estampida
[Centro de Visitantes] - Todo normal
[Laboratorio Genético] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Área de Velociraptores] - Todo normal
[Recinto de los triceratops] - Todo normal
[Centro de Visitantes] - Todo normal
[Laboratorio Genético] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Área de Velociraptores] - Pérdida de visibilidad
[Recinto de los triceratops] - Todo normal
[Centro de Visitantes] - Todo normal[Laboratorio Genético] - Falla del sis